# Day 3-03｜把 Track ID 放到球場上：ByteTrack-to-BEV 小專案

> 現在我們把前三天的元件串起來：YOLO 找球員、ByteTrack 維持 ID、球場 keypoints 估計 Homography、footpoint 轉成 BEV 座標。  
> 最終每一列資料都回答：**哪一個 track，在哪一個 frame，位於球場上的哪裡？**

## 我們會完成什麼

- 輸出原始視角與 BEV 並排的追蹤影片。
- 讀懂 `track_id -> bbox -> footpoint -> bev_xy` 的資料流。
- 將逐 frame 資料匯出 CSV，建立後續速度、路徑與站位分析的共同格式。
- 分辨 Homography 暫時失效與 tracking ID 失效是兩種不同問題。


## 前情提要（只保留這次需要的部分）

- Day 1：Homography 負責「相機座標 -> BEV 座標」。
- Day 2：player BBOX 的底邊中心作為 footpoint。
- Day 3-02：ByteTrack 為相鄰 frames 中的同一目標維持 `track_id`。

本 notebook 不再重算手動 IoU，也不重講 YOLO 欄位；我們專注在三者如何接成一條資料流。


## 工作坊流程

1. 設定影片、player detector、court keypoint model 與 BEV 規格。
2. 執行 tracking-to-BEV pipeline 並觀看左右對照影片。
3. 沿著一列資料檢查 `bbox_xyxy -> foot_x/foot_y -> bev_x/bev_y`。
4. 匯出 CSV，對每個 `track_id` 計算出現範圍與平均位置。
5. 記錄限制：ID switch、漏偵、球場 keypoint 不足與投影抖動。


In [ ]:
# 這一格要做什麼：準備課程環境並載入共用函式。
from pathlib import Path
import subprocess
import sys

COURSE_ROOT_HINT = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "course_setup.py").exists()),
    Path("/content/basketball_hackathon/course"),
)
if not (COURSE_ROOT_HINT / "src" / "course_setup.py").exists() and "google.colab" in sys.modules:
    COURSE_ROOT_HINT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "--depth", "1", "https://github.com/henry753951/basketball-hackathon-course.git", str(COURSE_ROOT_HINT)
    ], check=True)
if str(COURSE_ROOT_HINT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT_HINT))

from src.course_setup import bootstrap_course_repo  # noqa: E402

COURSE_ROOT = bootstrap_course_repo(COURSE_ROOT_HINT)


## Step 1｜設定整合流程的四種輸入

- 比賽影片：提供每個 frame。
- Player detector：輸出球員 BBOX。
- Court keypoint model：輸出球場固定點，用來估計 Homography。
- BEV 規格：定義鳥瞰球場的大小與線條位置。


In [ ]:
# 這一格要做什麼：指定影片、兩個模型、BEV 規格與輸出路徑。
import pandas as pd

from src.video_utils import display_video_in_notebook
from src.yolo_utils import (
    detector_model_path,
    preferred_court_keypoint_model_path,
    reference_videos,
    write_bytetrack_bev_video,
)

videos = reference_videos(COURSE_ROOT)
if len(videos) < 3:
    raise FileNotFoundError("assets/raw/reference_videos/ 至少需要三支參考影片。")

VIDEO_PATH = videos[2]
DETECTOR_PATH = detector_model_path(COURSE_ROOT)
COURT_MODEL_PATH = preferred_court_keypoint_model_path(COURSE_ROOT)
BEV_SPEC_PATH = COURSE_ROOT / "assets" / "samples" / "sample_bev_court.json"
OUTPUT_PATH = COURSE_ROOT / "assets" / "results" / "d3_03_bytetrack_bev.mp4"
OUT_CSV = COURSE_ROOT / "assets" / "results" / "d3_03_bytetrack_bev.csv"
START_FRAME = 30
MAX_FRAMES = 90

print("video:", VIDEO_PATH)
print("detector:", DETECTOR_PATH)
print("court model:", COURT_MODEL_PATH)
print("bev spec:", BEV_SPEC_PATH)
print("output video:", OUTPUT_PATH)
print("start frame:", START_FRAME)
print("max frames:", MAX_FRAMES)


## Step 2｜執行 `detection -> tracking -> footpoint -> Homography`

`write_bytetrack_bev_video()` 內部依序執行：

```text
frame
  -> player BBOX
  -> ByteTrack track_id
  -> BBOX bottom-center footpoint
  -> court keypoints -> Homography
  -> BEV coordinate
```

若當前 frame 的球場 keypoints 不足，程式會短暫沿用上一個可靠 Homography。這能減少 BEV 閃爍，但無法修復長時間看不到球場線的片段。


In [ ]:
# 這一格要做什麼：執行完整 tracking-to-BEV pipeline 並播放結果。
bev_video, records = write_bytetrack_bev_video(
    video_path=VIDEO_PATH,
    detector_path=DETECTOR_PATH,
    court_model_path=COURT_MODEL_PATH,
    bev_spec_path=BEV_SPEC_PATH,
    output_path=OUTPUT_PATH,
    max_frames=MAX_FRAMES,
    detector_conf=0.25,
    keypoint_conf=0.15,
    anchor_confidence=0.25,
    imgsz=960,
    start_frame=START_FRAME,
)

print("BEV video:", bev_video)
print("BEV json:", bev_video.with_suffix(".json"))
print("rows:", len(records))
display_video_in_notebook(bev_video, loop=True)


## Step 3｜沿著一列資料檢查座標如何改變

- `bbox_xyxy`：模型在相機畫面找到的矩形。
- `foot_x, foot_y`：矩形底邊中心，仍是相機像素座標。
- `bev_x, bev_y`：經 Homography 後的鳥瞰球場座標。
- `keypoint_count`：當時有多少球場點可支援 Homography；數量少時要提高警覺。


In [ ]:
# 這一格要做什麼：將 JSON records 整理成欄位固定的 DataFrame。
track_columns = [
    "frame",
    "track_id",
    "class_name",
    "confidence",
    "bbox_xyxy",
    "foot_x",
    "foot_y",
    "bev_x",
    "bev_y",
    "keypoint_count",
]
tracks = pd.DataFrame(records, columns=track_columns)
tracks.head(10)


## Step 4｜匯出可供研究分析的長表格

這種「每個 frame、每個 track 一列」的長表格可延伸為：

- 單一球員 BEV 路徑與移動距離。
- 速度、加速度與急停事件估計。
- 兩隊陣形、球員間距與攻守站位。
- 與投籃、傳球或回合事件的時間對齊。

注意：`track_id` 不是球員姓名或背號。若要跨片段辨認真實球員，還需要 Re-ID、背號辨識或人工校正。


In [ ]:
# 這一格要做什麼：匯出 CSV，並以 track_id 聚合出位置摘要。
tracks.to_csv(OUT_CSV, index=False)

if tracks.empty:
    summary = pd.DataFrame(columns=["track_id", "frames_seen", "first_frame", "last_frame", "mean_bev_x", "mean_bev_y"])
else:
    summary = (
        tracks.groupby("track_id", dropna=False)
        .agg(
            frames_seen=("frame", "count"),
            first_frame=("frame", "min"),
            last_frame=("frame", "max"),
            mean_bev_x=("bev_x", "mean"),
            mean_bev_y=("bev_y", "mean"),
        )
        .sort_values(["frames_seen", "first_frame"], ascending=[False, True])
        .reset_index()
    )

print("BEV csv:", OUT_CSV)
summary.head(15)


## 我們完成了什麼，也還缺什麼

Day 1–3 的位置資料主幹已完成：標註格式、YOLO detection、court-keypoint Homography、ByteTrack 與 BEV 路徑輸出。下一份 notebook 會加入隊伍分群，使用球員上半身的球衣色彩把 crops 分成兩組。

仍需記得：

- Homography 錯誤會讓位置投影偏移，但不一定改變 track ID。
- Tracking 錯誤會讓路徑接到錯的人，即使 BEV 投影本身正確。
- Team clustering 只能產生 Team A / B；它不知道真實隊名，也不等同背號或球員辨識。


## 本單元產出

- `assets/results/d3_03_bytetrack_bev.mp4`：原始視角與 BEV 並排影片。
- `assets/results/d3_03_bytetrack_bev.json`：逐 frame 原始紀錄。
- `assets/results/d3_03_bytetrack_bev.csv`：可供 pandas 與研究分析使用的長表格。
